In [0]:
# Imports
from effodata import ACDS, golden_rules, Joiner, Sifter, Equality, join_on
from kpi_metrics import KPI, AliasMetric, CustomMetric, AliasGroupby, get_metrics, available_metrics, Rollup, Cube
import pyspark.sql.functions as f
from pyspark.sql.types import *
from pyspark.sql.functions import when
from pyspark.sql.functions import when, lit
import re
import os
import sys
import time
import upc_input
import datetime as dt
import seg
from seg import profile
from pyspark.sql.window import Window
from poirot import SparkManager

#ignore strange depreciation warnings
from warnings import simplefilter 
simplefilter(action='ignore', category=DeprecationWarning)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
kpi = KPI(use_sample_mart = False, apply_privacy_filters = True)
acds = ACDS(use_sample_mart = False, apply_privacy_filters = True)

#### Closed Loop Summary Tab of KPF Dashboard.
- Contains only Gift, Open Loop, Account Funding, and Lottery campaigns.
- Aggregated Numbers at a overall campaign level for 2022 - 2025 campaigns, sourced from Media History and Campaign Info table.
- Will be automated with Absolute IROAS logic to contain both Adjusted and Absolute view numbers in the future.

<img src="./Closed_Loop_Summary_Tab_Sample.png" alt="Closed_Loop_Summary_Tab_Sample.png" width="1000"/>

In [0]:
# Code to generate closed loop campaign tab

# Reminder to use a Non-UC enabled cluster when pulling KPM metadata.
media_history_revamped = (
    spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_hist_revamped')
    .filter(f.col("campaign_id").rlike("^[0-9]+$"))
    .filter(f.col("camp_start_date").isNotNull())
).cache()
media_history_revamped.count() 

mmci = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_meas_campaign_info_v')

mmci = mmci.withColumn(
    'Fiscal_Quarter',
    f.concat(f.lit('Q'), f.substring(f.col('START_FIS_QUARTER_NAME'), 9, 1))
)
mmci = mmci.withColumn(
  "manufacturer", f.when(f.col("KPM_PROJECT_ID") == 93220, "Kroger Personal Finance").otherwise(f.col("manufacturer"))
)


# Rename and Join Media History and Campaign Availability Tables
# Use KPM Duplicated id
mhtv_mmci = media_history_revamped.join(
    mmci.select('kpm_duplicated_id', 'KROGER_START_WEEK', 'Fiscal_Start_Year', 'Fiscal_Quarter', 'target_id', 'channel', "SIGNED_OFF")
        .withColumnRenamed('Fiscal_Start_Year','year')
        .withColumnRenamed('Fiscal_Quarter','quarter')
        .withColumnRenamed('kpm_duplicated_id','campaign_id')
        .distinct(),
    ['campaign_id'],
    'inner'
)


# Hard Coding to remove incorrect product groups and discrepancies (from legacy code)
excluded_campaign_ids = [
    55062, 46606, 69524, 69520, 46773,
    64159, 63138, 68198, 43319, 44493, 45507
]
mhtv_mmci = (
    mhtv_mmci
    .filter(~f.col('product_group').like('%800000016326%'))
    .filter(~((f.col('campaign_id') == 49863) & (~f.lower(f.col('product_group')).like('%category%'))))
    .filter(~((f.col('campaign_id') != 49863) & (f.lower(f.col('product_group')).like('%category%'))))
    .filter(~f.col('product_group').like('%OL variable load%'))
    .filter(~((f.col('campaign_id') == 37539) & (f.col('adjusted_top_performer') == 'Not-Top-Performer_KRO')))
    .filter(~((f.col('campaign_id') == 89949) & (f.col('quarter') == 'Q1')))
    .filter(~f.col('campaign_id').isin(excluded_campaign_ids))
)

# Handle media history error case. campaign id 138141 XCM EMOD PUSH REM should be type XCM
mhtv_mmci = mhtv_mmci.withColumn(
    "campaign_type",
    when(f.col("campaign_id") == 138141, lit("XCM")).otherwise(f.col("campaign_type"))
)

mhtv_mmci = mhtv_mmci.filter(f.col("camp_start_date") >= "2023-01-01")
mhtv_mmci.display()

In [0]:
# Filters to obtain necessary KPF campaigns + filter by rom

mhtv_mmci = mhtv_mmci.withColumn(
  "manufacturer", f.when(f.col("campaign_id") == 93220, "Kroger Personal Finance").otherwise(f.col("manufacturer"))
)

kpf_mhtv_mmci_data = mhtv_mmci.filter(
    (f.col('KROGER_START_WEEK') >= '20220101') &
    (f.col("manufacturer").isin("Kroger Wallet", "Kroger Personal Finance")) &
    (f.col("channel")).isin(['Display Ad', 'Targeted Digital Coupon', 'Single Subject Email', 'Pandora', 'Pinterest', 'Push Notifications', 'Pre-Roll Video', 'Email Module']) & 
    (f.col("SIGNED_OFF") == "Y") & 
    (f.col("modality") == "All Modalities") &
    (f.col("rom") == "Kroger Only")
)


df_xcm = kpf_mhtv_mmci_data.filter(f.col("campaign_type") == "XCM")
df_tdc_sse_ect = kpf_mhtv_mmci_data.filter(f.col("campaign_type") != "XCM")

# 1. Updated Top Performer Logic: For NON XCMs, If a campaign has an offer attached, then 800's product groups will be top performers.
# If a campaign has NO offer attached, then the flag "Adjusted_Top_Performer" will be used to obtain top performers
# If a campaign has 800s product group attached but ALSO 4x (Regional), then use adjusted top performer flag (for example SSE Gift).

campaign_window = Window.partitionBy("campaign_id")
df_tdc_sse_ect = df_tdc_sse_ect.withColumn(
    "has_800", 
    f.max(f.when(f.col("product_group").like("800%"), 1).otherwise(0)).over(campaign_window)
).withColumn(
    "is_4x_exception",
    f.max(
        f.when(f.col("product_group").like("800%") & f.col("product_group").contains("4x"), 1).otherwise(0)
    ).over(campaign_window)
).filter(
    ((f.col("is_4x_exception") == 1) & (f.col("adjusted_top_performer") == "Top-Performer_KRO")) |
    ((f.col("is_4x_exception") == 0) & (f.col("has_800") == 1) & (f.col("product_group").like("800%"))) |
    ((f.col("is_4x_exception") == 0) & (f.col("has_800") == 0) & (f.col("adjusted_top_performer") == "Top-Performer_KRO"))
).drop("has_800", "is_4x_exception")

# 2. Updated Top Performer Logic: For XCMs, Adjusted Top Performers == "TOP-PERFORMER_KRO" will be Top Performers.
df_xcm = df_xcm.filter(f.col("adjusted_top_performer") == "Top-Performer_KRO")

# Could have done union after upstream calculations. Did it this way to QC intermediate outputs as well.
kpf_mhtv_mmci_data = df_tdc_sse_ect.union(df_xcm)
kpf_mhtv_mmci_data.display()



In [0]:
# Spring Easter Mothers Day Campaign had to be ran multiple times. We take the highest sales uplift total here for reporting (hardcoding one specific campaign instance)
kpf_mhtv_mmci_data = kpf_mhtv_mmci_data.withColumn(
    "sales_uplift_total",
    f.when(f.col("campaign_id") == 131511, f.lit(4596066.0)).otherwise(f.col("sales_uplift_total"))
)
kpf_mhtv_mmci_data.display()

In [0]:
# Saving out intermediate outputs for iroas automation 
kpf_mhtv_mmci_data.coalesce(1).write.mode("overwrite").option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/closed_loop_summary_tab_INTERMEDIATE.csv')

In [0]:
# Aggregation of data and upstream data calculations. Metrics should match its respective campaign_id's Wrap Report.
# NOTE: When joining media_meas_campaign_info and mhtv, XCM top performer rows appear multiple times (due to one line per channel).
# As a result, XCMs will be handled separately with averages instead of summation to ensure it matches wrap report.

# 5-18-2026 temp excluded some problematic campaign ids
kpf_mhtv_mmci_data_dedup = kpf_mhtv_mmci_data.dropDuplicates()

kpf_closed_loop_summary_tab_tdc_sse_ect = kpf_mhtv_mmci_data_dedup.filter(
    (f.col("campaign_type") != "XCM") & (~f.col("campaign_id").isin([125010]))
).groupby(
        "campaign_id", "job_id", "project_name", "campaign_type", "quarter", "year", "manufacturer", "camp_start_date", "camp_end_date").agg(
    # Campaign Detail Metrics
    f.max("test_hh_count").alias("HHs_reached"),
    f.sum("camp_cost").alias("Investment"),
    f.sum("camp_cost").alias("camp_cost"),
    f.sum("total_redemptions_cost").alias("Redemptions_Cost"),

    # Closed Loop Metrics
    f.sum("sales_test_total").alias("sales_test_total"),
    f.sum("sales_cont_total").alias("sales_cont_total"),
    f.sum("sales_uplift_total").alias("sales_uplift_total"),
    f.sum("sales_uplift_per_hh").alias("sales_uplift_per_hh"),

    f.sum("hhpen_test_total").alias("hhpen_test_total"),
    f.sum("hhpen_cont_total").alias("hhpen_cont_total"),
    f.sum("hhpen_uplift_total").alias("hhpen_uplift_total"),
    f.sum("hhpen_uplift_per_hh").alias("hhpen_uplift_per_hh"),

    f.sum("visits_test_total").alias("visits_test_total"),
    f.sum("visits_cont_total").alias("visits_cont_total"),
    f.sum("visits_uplift_total").alias("visits_uplift_total"),
    f.sum("visits_uplift_per_hh").alias("visits_uplift_per_hh"),

    f.sum("units_test_total").alias("units_test_total"),
    f.sum("units_cont_total").alias("units_cont_total"),
    f.sum("units_uplift_total").alias("units_uplift_total"),
    f.sum("units_uplift_per_hh").alias("units_uplift_per_hh"),

    # Engagement Metrics
    f.avg("clickthrough_rate").alias("clickthrough_rate"),
    f.avg("open_rate").alias("open_rate"),
    f.avg("redemption_rate").alias("redemption_rate"),
    f.avg("download_rate").alias("download_rate"),
    f.sum("total_downloads").alias("total_downloads"),
    f.sum("total_redemptions_visits").alias("total_redemptions_visits")

# Manual Upstream Calculations for Closed Loop Campaigns
).withColumn(
        "iROAS",
        f.col("sales_uplift_total") / f.col("camp_cost")
    ).withColumn(
        "aROAS",
        f.col("sales_test_total") / f.col("camp_cost")
    ).withColumn(
        "sales_uplift_pct",
        f.col("sales_uplift_total") / f.col("sales_cont_total")
    ).withColumn(
        "hhpen_uplift_pct",
        f.col("hhpen_uplift_total") / f.col("hhpen_cont_total")
    ).withColumn(
        "visits_uplift_pct",
        f.col("visits_uplift_total") / f.col("visits_cont_total")
    ).withColumn(
        "units_uplift_pct",
        f.col("units_uplift_total") / f.col("units_cont_total")
    ).withColumn(
        "average_gift_card_load_amount",
        f.col("sales_test_total") / f.col("units_test_total")
    )

kpf_closed_loop_summary_tab_tdc_sse_ect.display()

In [0]:
# Aggregation of data and upstream data calculations. Metrics should match its respective campaign_id's Wrap Report.

# NOTE: When joining media_meas_campaign_info and mhtv, XCM top performer rows appear multiple times (due to one line per channel).
# As a result, XCMs will be handled separately to ensure it matches wrap report.

# 5-18-2026 temp excluded some problematic campaign ids - change back later
kpf_closed_loop_summary_tab_xcm = kpf_mhtv_mmci_data.filter(
    (f.col("campaign_type") == "XCM") & (~f.col("campaign_id").isin([147403, 138143, 82541]))
).groupby(
    "campaign_id", "job_id", "project_name", "campaign_type", "quarter", "year", "manufacturer", "camp_start_date", "camp_end_date").agg(
    # Campaign Detail Metrics
    f.max("test_hh_count").alias("HHs_reached"),
    f.avg("camp_cost").alias("Investment"),
    f.avg("camp_cost").alias("camp_cost"),
    f.avg("total_redemptions_cost").alias("Redemptions_Cost"),

    # Closed Loop Metrics
    f.avg("sales_test_total").alias("sales_test_total"),
    f.avg("sales_cont_total").alias("sales_cont_total"),
    f.avg("sales_uplift_total").alias("sales_uplift_total"),
    f.avg("sales_uplift_per_hh").alias("sales_uplift_per_hh"),

    f.avg("hhpen_test_total").alias("hhpen_test_total"),
    f.avg("hhpen_cont_total").alias("hhpen_cont_total"),
    f.avg("hhpen_uplift_total").alias("hhpen_uplift_total"),
    f.avg("hhpen_uplift_per_hh").alias("hhpen_uplift_per_hh"),

    f.avg("visits_test_total").alias("visits_test_total"),
    f.avg("visits_cont_total").alias("visits_cont_total"),
    f.avg("visits_uplift_total").alias("visits_uplift_total"),
    f.avg("visits_uplift_per_hh").alias("visits_uplift_per_hh"),

    f.avg("units_test_total").alias("units_test_total"),
    f.avg("units_cont_total").alias("units_cont_total"),
    f.avg("units_uplift_total").alias("units_uplift_total"),
    f.avg("units_uplift_per_hh").alias("units_uplift_per_hh"),

    # Engagement Metrics
    f.avg("clickthrough_rate").alias("clickthrough_rate"),
    f.avg("open_rate").alias("open_rate"),
    f.avg("redemption_rate").alias("redemption_rate"),
    f.avg("download_rate").alias("download_rate"),
    f.sum("total_downloads").alias("total_downloads"),
    f.sum("total_redemptions_visits").alias("total_redemptions_visits")
        
# Manual Upstream Calculations for Closed Loop Campaigns
).withColumn(
        "iROAS",
        f.col("sales_uplift_total") / f.col("camp_cost")
    ).withColumn(
        "aROAS",
        f.col("sales_test_total") / f.col("camp_cost")
    ).withColumn(
        "sales_uplift_pct",
        f.col("sales_uplift_total") / f.col("sales_cont_total")
    ).withColumn(
        "hhpen_uplift_pct",
        f.col("hhpen_uplift_total") / f.col("hhpen_cont_total")
    ).withColumn(
        "visits_uplift_pct",
        f.col("visits_uplift_total") / f.col("visits_cont_total")
    ).withColumn(
        "units_uplift_pct",
        f.col("units_uplift_total") / f.col("units_cont_total")
    ).withColumn(
        "average_gift_card_load_amount",
        f.col("sales_test_total") / f.col("units_test_total")
    )

kpf_closed_loop_summary_tab_xcm.display()

In [0]:
kpf_closed_loop_summary_tab = kpf_closed_loop_summary_tab_tdc_sse_ect.union(kpf_closed_loop_summary_tab_xcm)

# Fill redemption_rate column as 0. We will manually handle this in absolute view automation.
if "redemption_rate" in kpf_closed_loop_summary_tab.columns:
    kpf_closed_loop_summary_tab = kpf_closed_loop_summary_tab.withColumn("Redemptions_Cost", f.lit(0))

kpf_closed_loop_summary_tab.display()


In [0]:
# Business Line Column (Tay's Logic that I converted to Python here)
kpf_closed_loop_summary_tab = kpf_closed_loop_summary_tab.withColumn(
    "business_line",
    f.when(
         f.upper(f.col("project_name")).contains("LOTT"), "Lottery"
    ).when(
        f.upper(f.col("project_name")).contains("OPEN LOOP") | 
        f.upper(f.col("project_name")).contains(" OL "), "Open Loop"
    ).when(
        f.upper(f.col("project_name")).contains("LOCAL") |
        f.upper(f.col("project_name")).contains("TDC KPF") |
        f.upper(f.col("project_name")).contains("TDC SFID") |
        f.upper(f.col("project_name")).contains("TDC SFPRJ") |
        f.upper(f.col("project_name")).contains("MCP") |
        f.upper(f.col("project_name")).contains("BULK") |
        f.upper(f.col("project_name")).contains("GIFT"), "Gift"
    ).when(
        f.upper(f.col("project_name")).contains("MONEY SERVICES") |
        f.upper(f.col("project_name")).contains(" MS "), "Money Services"
    ).when(
        f.upper(f.col("project_name")).contains(" PAY "), "Kroger Pay"
    ).when(
        f.upper(f.col("project_name")).contains("KROGER WALLET"), "Kroger Wallet"
    ).when(
        f.upper(f.col("project_name")).contains(" ACH "), "Debit"
    ).when(
        f.upper(f.col("project_name")).contains(" CICO "), "Account Funding"
    ).when(
        f.upper(f.col("project_name")).contains(" SBE "), "SBE"
    ).when(
        f.upper(f.col("project_name")).contains("CREDIT"), "Credit"
    ).when(
        f.upper(f.col("project_name")).contains(" REM "), "Gift"
    ).otherwise("")
)

kpf_closed_loop_summary_tab.display()

In [0]:
kpf_closed_loop_summary_tab = kpf_closed_loop_summary_tab.filter(f.col('camp_start_date') >= '2023-01-01')

# Hard Coding to remove some older 2022 campaigns with negative sales uplift %.
kpf_closed_loop_summary_tab = kpf_closed_loop_summary_tab.filter(~f.col("campaign_id").isin([68198, 64159, 63138, 45507, 44493, 43319]))

# Hard Coding a small few campaigns where metadata is not reflecting wrap report: 138143
kpf_closed_loop_summary_tab = kpf_closed_loop_summary_tab.withColumn(
    "sales_uplift_pct",
    f.when(f.col("campaign_id") == 138143, f.lit(0.00)).otherwise(f.col("sales_uplift_pct"))
).withColumn(
    "hhpen_uplift_pct",
    f.when(f.col("campaign_id") == 138143, f.lit(0.0108)).otherwise(f.col("hhpen_uplift_pct"))
).withColumn(
    "visits_uplift_pct",
    f.when(f.col("campaign_id") == 138143, f.lit(0.0078)).otherwise(f.col("visits_uplift_pct"))
)

# Hard Coding a small few campaigns where metadata is not reflecting wrap report: 147403
kpf_closed_loop_summary_tab = kpf_closed_loop_summary_tab.withColumn(
    "sales_uplift_pct",
    f.when(f.col("campaign_id") == 147403, f.lit(0.00)).otherwise(f.col("sales_uplift_pct"))
).withColumn(
    "hhpen_uplift_pct",
    f.when(f.col("campaign_id") == 147403, f.lit(0.0327)).otherwise(f.col("hhpen_uplift_pct"))
).withColumn(
    "visits_uplift_pct",
    f.when(f.col("campaign_id") == 147403, f.lit(0.0110)).otherwise(f.col("visits_uplift_pct"))
).withColumn(
    "units_uplift_pct",
    f.when(f.col("campaign_id") == 147403, f.lit(0.0119)).otherwise(f.col("visits_uplift_pct"))
)

# Hard Coding a small few campaigns where metadata is not reflecting wrap report: 82541
kpf_closed_loop_summary_tab = kpf_closed_loop_summary_tab.withColumn(
    "sales_uplift_pct",
    f.when(f.col("campaign_id") ==  82541, f.lit(0.00)).otherwise(f.col("sales_uplift_pct"))
).withColumn(
    "hhpen_uplift_pct",
    f.when(f.col("campaign_id") ==  82541, f.lit(0.0374)).otherwise(f.col("hhpen_uplift_pct"))
).withColumn(
    "visits_uplift_pct",
    f.when(f.col("campaign_id") ==  82541, f.lit(0.0168)).otherwise(f.col("visits_uplift_pct"))
).withColumn(
    "units_uplift_pct",
    f.when(f.col("campaign_id") ==  82541, f.lit(0.0348)).otherwise(f.col("visits_uplift_pct"))
)


kpf_closed_loop_summary_tab.display()

#### Dollar Off Offers: Handled here
- Some campaigns have monetary $ off rather than fuel points. As a result, we need to use the redemptions column in mhtv as dollar value and manually calculate absolute #s, rather than utilizing the fuel points methodology.

In [0]:
# ABS numbers for $ off redemption cost campaigns 
kpf_open_loop_dollar_off_campaigns = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_OL_dollar_off')

# Join with ABS numbers
kpf_closed_loop_dollar_off_summary_with_abs = kpf_closed_loop_summary_tab.join(kpf_open_loop_dollar_off_campaigns, on="campaign_id", how="inner")

# Drop current "Redemptions_Cost" and rename "new_redemption_cost" as "Redemptions_Cost"
if "Redemptions_Cost" in kpf_closed_loop_dollar_off_summary_with_abs.columns:
    kpf_closed_loop_dollar_off_summary_with_abs = kpf_closed_loop_dollar_off_summary_with_abs.drop("Redemptions_Cost")
if "new_redemption_cost" in kpf_closed_loop_dollar_off_summary_with_abs.columns:
    kpf_closed_loop_dollar_off_summary_with_abs = kpf_closed_loop_dollar_off_summary_with_abs.withColumnRenamed("new_redemption_cost", "Redemptions_Cost")

kpf_closed_loop_dollar_off_summary_with_abs.display()

#### Handling Single Channel 2026+
- Since XCM is being deprecated at the end of 2025, I split up REM SSE PUSH and Offsite campaigns into single channel. Handling here, although in the future I should think of a better way to handle this

In [0]:
rem_sse_push_offer_codes_offsite = spark.read.parquet(f'abfss://tmkrprtnrs-users@sa8451krprtnrdev.dfs.core.windows.net/a136627/META_DATA_INFO_OFFSITE')
rem_sse_push_offer_codes_offsite.display()

In [0]:
rem_sse_push_offer_codes = spark.read.parquet(f'abfss://tmkrprtnrs-users@sa8451krprtnrdev.dfs.core.windows.net/a136627/META_DATA_INFO')
rem_sse_push_offer_codes.display()

In [0]:
kpf_closed_loop_absolute_nums = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_all_camp_types').withColumnRenamed("kpm_duplicated_id", "campaign_id")
kpf_closed_loop_absolute_nums.display()

In [0]:
# Reading in fuel point Absolute Numbers from Automation File
kpf_closed_loop_absolute_nums = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_all_camp_types').withColumnRenamed("kpm_duplicated_id", "campaign_id")

# Join with rem_sse_push_offer_codes where campaign_id = KPM_PROJECT_ID, keep CHANNEL and PROJECT_NAME from rem_sse_push_offer_codes
rem_sse_push_offer_codes_sel = rem_sse_push_offer_codes.select("KPM_PROJECT_ID", "KPM_DUPLICATED_ID", "CHANNEL")
kpf_closed_loop_absolute_nums_joined = kpf_closed_loop_absolute_nums.join(
    rem_sse_push_offer_codes_offsite.select("KPM_DUPLICATED_ID", "KPM_PROJECT_ID", "CHANNEL", "SALESFORCE_ID"),
    kpf_closed_loop_absolute_nums["campaign_id"] == rem_sse_push_offer_codes_offsite["KPM_PROJECT_ID"],
    how="left"
)

# Replace campaign_id with KPM_DUPLICATED_ID if not null
kpf_closed_loop_absolute_nums_joined = kpf_closed_loop_absolute_nums_joined.withColumn(
    "campaign_id",
    f.when(f.col("KPM_DUPLICATED_ID").isNotNull(), f.col("KPM_DUPLICATED_ID")).otherwise(f.col("campaign_id"))
)

# Deduplicate at the end and drop kpm_duplicated_id and project_name
kpf_closed_loop_absolute_nums_joined = kpf_closed_loop_absolute_nums_joined.dropDuplicates()
kpf_closed_loop_absolute_nums_joined.display()

In [0]:
# Join with ABS numbers
kpf_closed_loop_summary_with_abs = kpf_closed_loop_summary_tab.join(kpf_closed_loop_absolute_nums_joined, on="campaign_id", how="left")

if "CHANNEL" in kpf_closed_loop_summary_with_abs.columns:
    kpf_closed_loop_summary_with_abs = (
        kpf_closed_loop_summary_with_abs
        .withColumn("campaign_type", f.coalesce(f.col("CHANNEL"), f.col("campaign_type")))
        #.drop("CHANNEL", "KPM_DUPLICATED_ID", "KPM_PROJECT_ID")
    )

# Replace "Push Notifications" with "PUSH" and "Single Subject Email" with "SSE" in campaign_type
kpf_closed_loop_summary_with_abs = kpf_closed_loop_summary_with_abs.replace(
    {"Push Notifications": "PUSH", "Single Subject Email": "SSE", "Display Ad": "DISPLAY_AD"}, subset=["campaign_type"]
)

kpf_closed_loop_summary_with_abs.display()

In [0]:
# Coalesce nulls in new_redemption_cost to 0
if "new_redemption_cost" in kpf_closed_loop_summary_with_abs.columns:
    kpf_closed_loop_summary_with_abs = kpf_closed_loop_summary_with_abs.withColumn(
        "new_redemption_cost", f.coalesce(f.col("new_redemption_cost"), f.lit(0))
    )

# Drop current "Redemptions_Cost" and rename "new_redemption_cost" as "Redemptions_Cost"
if "Redemptions_Cost" in kpf_closed_loop_summary_with_abs.columns:
    kpf_closed_loop_summary_with_abs = kpf_closed_loop_summary_with_abs.drop("Redemptions_Cost")
if "new_redemption_cost" in kpf_closed_loop_summary_with_abs.columns:
    kpf_closed_loop_summary_with_abs = kpf_closed_loop_summary_with_abs.withColumnRenamed("new_redemption_cost", "Redemptions_Cost")

# Drop kpm_project_id, kpm_duplicated_id, and channel
cols_to_drop = [c for c in ["KPM_PROJECT_ID", "KPM_DUPLICATED_ID", "CHANNEL", "SALESFORCE_ID"] if c in kpf_closed_loop_summary_with_abs.columns]
if cols_to_drop:
    kpf_closed_loop_summary_with_abs = kpf_closed_loop_summary_with_abs.drop(*cols_to_drop)

kpf_closed_loop_summary_with_abs.display()

In [0]:
# Replace NULL fuel point iroas rows with dollar off calculated rows
ol_dollar_off_camp_ids = kpf_closed_loop_dollar_off_summary_with_abs.select("campaign_id").distinct()

kpf_summary_cleaned = kpf_closed_loop_summary_with_abs.join(
    f.broadcast(ol_dollar_off_camp_ids), 
    on="campaign_id", 
    how="left_anti"
)

kpf_closed_loop_summary_with_abs_final = kpf_summary_cleaned.unionByName(kpf_closed_loop_dollar_off_summary_with_abs)
kpf_closed_loop_summary_with_abs_final.display()

In [0]:
# Filter to include only open loop, gift, account funding, and lottery campaigns for closed loop
kpf_closed_loop_summary_with_abs_final = kpf_closed_loop_summary_with_abs_final.filter(
    f.col("business_line").isin("Open Loop", "Gift", "Account Funding", "Lottery")
)
kpf_closed_loop_summary_with_abs_final.display()

#### No Offer Campaigns: Handled here.
- If a campaign does not exist in the mmoi table and does not have an offer barcode, we would convert redemption cost to 0 and manually calculate absolute #s.

In [0]:
# Grab recent (2024 onwards) campaigns where abs #s are NULL (not $ amount OR fuel points means no redemption)
kpf_closed_loop_summary_with_abs_final_relevant = kpf_closed_loop_summary_with_abs_final.filter(
    (f.col("year") >= 2024) & (
        f.col("abs_sales_uplift").isNull() | f.col("abs_iroas").isNull()
    ) & f.col("campaign_type").isin("EMOD", "SSE")
).withColumnRenamed("campaign_id", "kpm_duplicated_id")
kpf_closed_loop_summary_with_abs_final_relevant.display()

In [0]:
# Keep only campaigns with no offer, joining on mmoi.KPM_PROJECT_ID and kpf_closed_loop_summary_with_abs_final_relevant.kpm_duplicated_id
mmoi = spark.read.parquet(f'abfss://landingzone@sa8451entlakegrnprd.dfs.core.windows.net/mart/comms/prd/measurement/MEDIA_MEAS_OFFER_INFO')
kpf_closed_loop_summary_with_abs_no_offer = kpf_closed_loop_summary_with_abs_final_relevant.join(
    f.broadcast(mmoi.select("KPM_PROJECT_ID").distinct()), 
    kpf_closed_loop_summary_with_abs_final_relevant.kpm_duplicated_id == mmoi.KPM_PROJECT_ID,
    how="left_anti"
)
kpf_closed_loop_summary_with_abs_no_offer.display()

In [0]:
# Make new working cost based on campaign_type
kpf_closed_loop_summary_with_abs_no_offer = kpf_closed_loop_summary_with_abs_no_offer.withColumn(
    "multiplier",
    f.when(f.col("campaign_type") == "DISP", f.lit(0.3520))
    .when(f.col("campaign_type") == "EMOD", f.lit(0.015))
    .when(f.col("campaign_type") == "PAND", f.lit(0.741))
    .when(f.col("campaign_type") == "PINT", f.lit(0.663))
    .when(f.col("campaign_type") == "PRV", f.lit(0.3960))
    .when(f.col("campaign_type") == "PUSH", f.lit(0.038))
    .when(f.col("campaign_type") == "SSE", f.lit(0.3390))
    .when(f.col("campaign_type") == "TDC", f.lit(0.141))
    .otherwise(f.lit(0))
).withColumn(
    "working_cost",
    f.col("camp_cost") * f.col("multiplier")
)
kpf_closed_loop_summary_with_abs_no_offer.display()

In [0]:
# adj_total_cost = working_cost + camp cost
# abs_total_cost = working_cost + redemption_cost
# redemption_cost = abs_total_cost - working_cost. redemption_cost AKA redemption $ amount
kpf_closed_loop_summary_with_abs_no_offer_agg = (kpf_closed_loop_summary_with_abs_no_offer
    .withColumn("adj_total_cost", f.col("working_cost") + f.col("camp_cost"))
    .withColumn("abs_total_cost", f.col("working_cost") + f.col("Redemptions_Cost"))
    .withColumn('adj_sales_uplift', f.col("sales_uplift_total").cast('Integer'))
    .withColumn("abs_sales_uplift", f.round(f.col("sales_uplift_total") - f.col("Redemptions_Cost"), 2))
    .withColumn("abs_sales_test_earned", f.round(f.col("sales_test_total") - f.col("Redemptions_Cost"), 2))
    .withColumn("adj_iroas", f.round(f.col("sales_uplift_total") / f.col("adj_total_cost"), 2))
    .withColumn("abs_iroas", f.round(f.col("abs_sales_uplift") / f.col("abs_total_cost"), 2))
    .withColumn("adj_aroas", f.round(f.col("sales_test_total") / f.col("adj_total_cost"), 2))
    .withColumn("abs_aroas", f.round(f.col("abs_sales_test_earned") / f.col("abs_total_cost"), 2))
)

kpf_closed_loop_summary_with_abs_no_offer_agg.select(
    "kpm_duplicated_id", "working_cost", "adj_total_cost", "abs_total_cost", "abs_sales_uplift", 
    "abs_sales_test_earned", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas"
).display()

In [0]:
# Make negative #s to 0, since we do not report negative #s.
cols_to_zero_neg = [
    "working_cost", "adj_sales_uplift", "abs_sales_uplift", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost", "abs_sales_test_earned"
]

for c in cols_to_zero_neg:
    kpf_closed_loop_summary_with_abs_no_offer_agg = kpf_closed_loop_summary_with_abs_no_offer_agg.withColumn(c, f.when(f.col(c) < 0, 0).otherwise(f.col(c)))

kpf_closed_loop_summary_with_abs_no_offer_final = kpf_closed_loop_summary_with_abs_no_offer_agg.withColumnRenamed("kpm_duplicated_id", "campaign_id") \
    .drop("multiplier", "adj_sales_total")

kpf_closed_loop_summary_with_abs_no_offer_final.display()

In [0]:
# Replace NULL fuel point iroas rows with dollar off calculated rows
no_offer_attached_ids = kpf_closed_loop_summary_with_abs_no_offer_final.select("campaign_id").distinct()

kpf_summary_cleaned = kpf_closed_loop_summary_with_abs_final.join(
    f.broadcast(no_offer_attached_ids), 
    on="campaign_id", 
    how="left_anti"
)

kpf_closed_loop_summary_with_abs_final = kpf_summary_cleaned.unionByName(
    kpf_closed_loop_summary_with_abs_no_offer_final
)

kpf_closed_loop_summary_with_abs_final.display()

In [0]:
# Ad Hoc Check to see if rr/dr should be median or mean
'''
dr_rr_distribution_check = kpf_closed_loop_summary_with_abs_final.select(
    f.skewness("download_rate").alias("download_skew"),
    f.skewness("redemption_rate").alias("redemption_skew")
).display()

# Calculate Mean and Standard Deviation
stats = kpf_closed_loop_summary_with_abs_final.select(
    f.avg("redemption_rate").alias("avg_rate"),
    f.stddev("redemption_rate").alias("std_dev")
).collect()[0]

mu = stats['avg_rate']
sigma = stats['std_dev']
outliers = kpf_closed_loop_summary_with_abs_final.filter((f.col("redemption_rate") > mu + 3*sigma) | 
                     (f.col("redemption_rate") < mu - 3*sigma))

print(f"Number of extreme outliers: {outliers.count()}")
outliers.display()
'''

In [0]:
'''
import matplotlib.pyplot as plt
import scipy.stats as stats
import numpy as np

df = kpf_closed_loop_summary_with_abs_final.select("download_rate", "redemption_rate").dropna().toPandas()

plt.figure(figsize=(12, 10))

plt.subplot(2, 2, 1)
stats.probplot(df["download_rate"], dist="norm", plot=plt)
plt.title("QQ Plot: Download Rate")

plt.subplot(2, 2, 2)
stats.probplot(df["redemption_rate"], dist="norm", plot=plt)
plt.title("QQ Plot: Redemption Rate")

plt.subplot(2, 2, 3)
mu = df["download_rate"].mean()
sigma = df["download_rate"].std()
x = np.linspace(df["download_rate"].min(), df["download_rate"].max(), 100)
plt.hist(df["download_rate"], bins=30, density=True, alpha=0.6, color='g')
plt.plot(x, stats.norm.pdf(x, mu, sigma), 'k', linewidth=2)
plt.title("Bell Curve: Download Rate")

plt.subplot(2, 2, 4)
mu = df["redemption_rate"].mean()
sigma = df["redemption_rate"].std()
x = np.linspace(df["redemption_rate"].min(), df["redemption_rate"].max(), 100)
plt.hist(df["redemption_rate"], bins=30, density=True, alpha=0.6, color='b')
plt.plot(x, stats.norm.pdf(x, mu, sigma), 'k', linewidth=2)
plt.title("Bell Curve: Redemption Rate")

plt.tight_layout()
plt.show()
'''

#### Final Processing: Pivot to long format for ingestion into PBI

In [0]:
# Created melt function to unpivot dataframe from wide to long format
def melt(df, id_vars, value_vars, var_name = "variable", value_name = "value"):
    n = len(value_vars)
    expr = ", ".join([f"'{c}', {c}" for c in value_vars])
    return df.selectExpr(
        *id_vars,
        f"stack({n}, {expr}) as ({var_name}, {value_name})"
    )

In [0]:
# Melt from wide to long format with a metric column and its corresponding value as another column
kpf_non_metric_cols = [
    'campaign_id', "job_id", "project_name", "campaign_type", "quarter", "year", "manufacturer",
    "camp_start_date", "camp_end_date", "business_line"
]

kpf_closed_loop_summary_tab_cols = kpf_closed_loop_summary_with_abs_final.columns 
kpf_closed_loop_sumary_metric_cols = [col for col in kpf_closed_loop_summary_with_abs_final.columns if col not in kpf_non_metric_cols]

kpf_final_prepared = kpf_closed_loop_summary_with_abs_final.select(
    *kpf_non_metric_cols, # Keep your ID columns as they are
    *[f.col(c).cast("double").alias(c) for c in kpf_closed_loop_sumary_metric_cols] # Force all metrics to double
)

# 3. NOW run the melt on the fully prepared dataframe
kpf_closed_loop_summary_tab_melted = melt(
    kpf_final_prepared, 
    id_vars=kpf_non_metric_cols,
    value_vars=kpf_closed_loop_sumary_metric_cols, 
    var_name="metric", 
    value_name="value"
)


'''
kpf_closed_loop_summary_tab_melted = kpf_closed_loop_summary_tab_melted.filter(
    f.col("business_line").isin("Open Loop", "Gift", "Account Funding", "Lottery")
).withColumn(
    "value", f.round(f.col("value"), 2)
)
'''
kpf_closed_loop_summary_tab_melted.display()

In [0]:
# Write to P&LS data location and output for testing
kpf_closed_loop_summary_tab_melted.coalesce(1).write.mode("overwrite").option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/closed_loop_summary_tab.csv')

kpf_closed_loop_summary_tab_melted_csv = spark.read.option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/closed_loop_summary_tab.csv')
kpf_closed_loop_summary_tab_melted_csv.display()